# LeetCode 4061 — Minimum Queen Moves to Reach Target bằng Python

Link đề: https://leetcode.com/problems/minimum-queen-moves-to-reach-target/ · Độ khó: **Easy** · Chủ đề: Array, Math

Skeleton LeetCode:

```python
class Solution:
    def minQueenMoves(self, source: List[int], target: List[int]) -> int:
        ...
```

Ghi chú: đề và các ví dụ lấy từ đề gốc (Biweekly Contest 192, câu 1); mọi đáp án trong notebook đều được kiểm chứng bằng code, kể cả bằng BFS độc lập trên **toàn bộ** 64 × 64 cặp ô của bàn cờ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: bàn cờ `8 × 8` **trống**, quân **Hậu (queen)** mỗi nước đi **một hoặc nhiều ô** theo **một hàng, một cột hoặc một đường chéo**; tìm **số nước đi ít nhất** từ `source` tới `target`.
2. Biết nhận ra quân Hậu "tới được" ô đích **trong một nước** khi cùng hàng (`sr == tr`), cùng cột (`sc == tc`) hoặc cùng đường chéo (`|sr − tr| == |sc − tc|`).
3. Đi từ **BFS tổng quát trên bàn cờ** → **thử mọi ô trung gian** → **phân tích trường hợp** (`0`, `1` hoặc `2`) → **biểu thức một dòng**, và hiểu **vì sao đáp án không bao giờ vượt quá 2**.
4. Làm quen **tọa độ 1-indexed**, `abs`, tuple, `set`, và BFS bằng `deque` trên lưới.
5. Tránh các lỗi kinh điển: quên trường hợp `0` (cùng ô), quên đường chéo phụ, nhầm sang cách đi của **vua** (khoảng cách Chebyshev) hay **Manhattan**, và nghĩ rằng đáp án có thể lớn hơn 2.

## 2. Đề bài & ví dụ

Có một bàn cờ `8 × 8` **trống**, hàng và cột được đánh số từ `1`. Cho `source = [sr, sc]` là vị trí xuất phát của quân Hậu và `target = [tr, tc]` là vị trí đích.

Trong **một nước đi**, quân Hậu đi **một hoặc nhiều ô** dọc theo **một hàng, một cột hoặc một đường chéo**, và phải nằm trong bàn cờ.

Trả về **số nước đi nhỏ nhất** để quân Hậu **đáp xuống đúng** ô `target`.

**Ví dụ 1**

- Input: `source = [8,1]`, `target = [1,8]`
- Output: `1`
- Giải thích: một nước đi chéo đưa quân Hậu thẳng từ `(8, 1)` tới `(1, 8)`.

**Ví dụ 2**

- Input: `source = [4,2]`, `target = [1,3]`
- Output: `2`
- Giải thích: quân Hậu đi từ `(4, 2)` tới `(4, 3)` (đi ngang), rồi từ `(4, 3)` tới `(1, 3)` (đi dọc): 2 nước.

**Ví dụ 3**

- Input: `source = [1,1]`, `target = [1,1]`
- Output: `0`
- Giải thích: quân Hậu đã ở ô đích nên không cần nước nào.

**Ràng buộc:** `source == [sr, sc]`; `target == [tr, tc]`; `1 <= sr, sc, tr, tc <= 8`.

In [1]:
from typing import List
import random
from collections import deque

examples = [
    ([8, 1], [1, 8], 1),
    ([4, 2], [1, 3], 2),
    ([1, 1], [1, 1], 0),
]
for source, target, expected in examples:
    print(f"source={source}, target={target}, kỳ vọng {expected}")

source=[8, 1], target=[1, 8], kỳ vọng 1
source=[4, 2], target=[1, 3], kỳ vọng 2
source=[1, 1], target=[1, 1], kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Tọa độ `(hàng, cột)` và gán nhiều biến cùng lúc

Mỗi ô của bàn cờ là một cặp `(r, c)` với `1 <= r, c <= 8`. Đề cho `source` và `target` là list `[r, c]`. Ta "bung" list thành hai biến bằng `sr, sc = source`.

In [2]:
source = [4, 2]
target = [1, 3]
sr, sc = source
tr, tc = target
print("source:", sr, sc, "| target:", tr, tc)
print("chênh lệch hàng dr =", tr - sr, ", chênh lệch cột dc =", tc - sc)

source: 4 2 | target: 1 3
chênh lệch hàng dr = -3 , chênh lệch cột dc = 1


### 3.2. Quân Hậu đi theo ba kiểu đường

Từ ô `(sr, sc)` quân Hậu tới được ô `(tr, tc)` **trong một nước** khi và chỉ khi:

- **cùng hàng:** `sr == tr`;
- **cùng cột:** `sc == tc`;
- **cùng đường chéo:** hai ô cách nhau cùng số hàng và số cột, `|sr − tr| == |sc − tc|`. Chéo chính (`↘`) có `sr − sc` không đổi; chéo phụ (`↙`) có `sr + sc` không đổi; điều kiện `|dr| == |dc|` gộp cả hai.

Vì bàn cờ **trống** nên không có quân nào cản đường.

In [3]:
def one_move(source, target):
    """Quân Hậu đi 1 nước từ source tới target (hai ô khác nhau)?"""
    sr, sc = source
    tr, tc = target
    return sr == tr or sc == tc or abs(sr - tr) == abs(sc - tc)


tests = [([8, 1], [1, 8]), ([4, 2], [4, 3]), ([4, 3], [1, 3]), ([4, 2], [1, 3]), ([1, 2], [2, 1])]
for a, b in tests:
    print(a, "->", b, "| 1 nước?", one_move(a, b))

[8, 1] -> [1, 8] | 1 nước? True
[4, 2] -> [4, 3] | 1 nước? True
[4, 3] -> [1, 3] | 1 nước? True
[4, 2] -> [1, 3] | 1 nước? False
[1, 2] -> [2, 1] | 1 nước? True


### 3.3. Giá trị tuyệt đối `abs`

`abs(x)` trả về `|x|`: `abs(-3) = 3`, `abs(3) = 3`. Ta cần nó vì hai ô có thể ở hai phía khác nhau (chênh lệch hàng âm hoặc dương) nhưng khoảng cách "đếm ô" luôn không âm.

In [4]:
print(abs(-3), abs(3), abs(0))
print(abs(1 - 8) == abs(8 - 1))                         # True: |dr| không phụ thuộc chiều
print("chéo chính & chéo phụ có |dr| == |dc|:", abs(3 - 6) == abs(5 - 2))

3 3 0
True
chéo chính & chéo phụ có |dr| == |dc|: True


### 3.4. `set`, `deque` và tuple làm khóa

BFS (duyệt theo tầng) cần một **hàng đợi** để lấy đỉnh theo thứ tự và một **tập/dict** ghi nhớ ô đã đi qua. `collections.deque` có `append` (thêm cuối) và `popleft` (lấy đầu) đều `O(1)`. Ô `(r, c)` dạng **tuple** là hashable, dùng được làm khóa của `dict` hoặc phần tử của `set`; list thì không.

In [5]:
q = deque([(4, 2)])
seen = {(4, 2): 0}                       # ô -> số nước đi đã dùng
cur = q.popleft()
print("lấy ra:", cur, "| còn lại trong hàng đợi:", list(q))
print((4, 3) in seen, (4, 2) in seen)    # False True
try:
    {[4, 2]: 0}                          # list không hashable
except TypeError as e:
    print("TypeError:", e)

lấy ra: (4, 2) | còn lại trong hàng đợi: []
False True
TypeError: unhashable type: 'list'


### 3.5. Duyệt một tia theo hướng `(dr, dc)` cho tới khi hết bàn cờ

Quân Hậu đi theo 8 hướng: `(±1, 0)`, `(0, ±1)`, `(±1, ±1)`. Từ ô `(r, c)` ta cứ cộng `(dr, dc)` liên tiếp, mỗi lần được thêm một ô tới được, **dừng khi ra khỏi** bàn cờ `1..8`.

In [6]:
DIRECTIONS = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]


def one_move_squares(r, c, size=8):
    """Mọi ô tới được từ (r, c) trong một nước (bàn cờ trống)."""
    result = []
    for dr, dc in DIRECTIONS:
        nr, nc = r + dr, c + dc
        while 1 <= nr <= size and 1 <= nc <= size:
            result.append((nr, nc))
            nr, nc = nr + dr, nc + dc
    return result


print(len(one_move_squares(1, 1)), "ô từ góc (1,1)")          # 21
print(len(one_move_squares(4, 4)), "ô từ gần giữa (4,4)")     # 27
print(sorted(one_move_squares(4, 2))[:6], "...")

21 ô từ góc (1,1)
27 ô từ gần giữa (4,4)
[(1, 2), (1, 5), (2, 2), (2, 4), (3, 1), (3, 2)] ...


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy xét ví dụ 2: `(4, 2) → (1, 3)`.

- Cùng hàng? `4 ≠ 1`. Cùng cột? `2 ≠ 3`. Cùng đường chéo? `|4 − 1| = 3`, `|2 − 3| = 1`, `3 ≠ 1`. Vậy **không** tới được trong 1 nước.
- Nhưng ta chọn một ô trung gian là "góc chữ L": đi **ngang** từ `(4, 2)` tới `(4, 3)` (cùng hàng 4), rồi đi **dọc** từ `(4, 3)` tới `(1, 3)` (cùng cột 3). Tổng **2 nước**.

Ý này đúng với **mọi** cặp ô khác nhau và không cùng đường: đi ngang tới cột của đích (`(sr, tc)`), rồi đi dọc tới đích. Hai ô `(sr, sc)` và `(sr, tc)` khác nhau (vì `sc ≠ tc`), và `(sr, tc)` khác `(tr, tc)` (vì `sr ≠ tr`); cả hai nước đều hợp lệ. Vậy đáp án luôn **≤ 2**.

Còn lại chỉ cần phân loại: cùng ô → `0`; cùng đường → `1`; còn lại → `2`.

In [7]:
source, target = (4, 2), (1, 3)
sr, sc = source
tr, tc = target
corner = (sr, tc)                     # góc chữ L: cùng hàng với source, cùng cột với target
print("ô trung gian:", corner)
print("source -> corner 1 nước?", one_move(source, corner))
print("corner -> target 1 nước?", one_move(corner, target))

ô trung gian: (4, 3)
source -> corner 1 nước? True
corner -> target 1 nước? True


## 5. Cách 1 — Brute Force / baseline: BFS trên bàn cờ

**Ý tưởng.** "Số nước đi ít nhất" trên đồ thị không trọng số là bài toán **đường đi ngắn nhất** nên ta dùng **BFS** (duyệt theo tầng). Đỉnh là các ô của bàn cờ `8 × 8`. Từ mỗi ô có cạnh tới mọi ô tới được trong một nước. Bắt đầu từ `source` với khoảng cách `0`; lấy từng ô ra khỏi hàng đợi, thêm các ô kề chưa thăm với khoảng cách `+1`. Gặp `target` thì dừng.

Cách này **không cần biết quy luật nào** của quân Hậu: nó chỉ làm đúng theo luật đi, nên đáng tin cậy dùng làm chuẩn đối chiếu.

In [8]:
def min_moves_bfs(source: list[int], target: list[int]) -> int:
    start = tuple(source)
    goal = tuple(target)
    dist = {start: 0}
    queue = deque([start])
    while queue:
        r, c = queue.popleft()
        if (r, c) == goal:
            return dist[(r, c)]
        for dr, dc in DIRECTIONS:
            nr, nc = r + dr, c + dc
            while 1 <= nr <= 8 and 1 <= nc <= 8:            # một nước: đi tia tới khi hết bàn cờ
                if (nr, nc) not in dist:
                    dist[(nr, nc)] = dist[(r, c)] + 1
                    queue.append((nr, nc))
                nr, nc = nr + dr, nc + dc
    return -1            # không xảy ra: bàn cờ liên thông


for source, target, expected in examples:
    print(f"source={source}, target={target} -> {min_moves_bfs(source, target)} (kỳ vọng {expected})")

source=[8, 1], target=[1, 8] -> 1 (kỳ vọng 1)
source=[4, 2], target=[1, 3] -> 2 (kỳ vọng 2)
source=[1, 1], target=[1, 1] -> 0 (kỳ vọng 0)


**Độ phức tạp (Cách 1).** Bàn cờ có `64` đỉnh; từ mỗi đỉnh duyệt 8 tia, mỗi tia tối đa 7 ô, tức tối đa `56` cạnh. Với bàn cờ cạnh `N`: **Time `O(N³)`** (`N²` đỉnh × `O(N)` cạnh mỗi đỉnh), **Space `O(N²)`**. Với `N = 8` là hằng số nhỏ, nhưng viết nhiều code hơn cần thiết.

**Nút thắt.** BFS làm việc thừa vì ta đã biết trước cấu trúc rất đặc biệt của bài: **luôn trả lời ≤ 2** và chỉ cần xét vài điều kiện. Ta sẽ rút gọn về `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

Có đúng **ba** giá trị khả dĩ:

| trường hợp | điều kiện | đáp án |
|---|---|---|
| đã ở đích | `source == target` | `0` |
| tới ngay trong 1 nước | cùng hàng, cùng cột hoặc cùng đường chéo | `1` |
| còn lại | không cùng đường nào | `2` |

**Vì sao "còn lại" luôn là 2?** Nếu hai ô khác nhau và không cùng hàng, cột lẫn đường chéo thì ô `(sr, tc)` là ô trung gian hợp lệ (nằm trong bàn cờ vì dùng toạ độ có sẵn), khác cả hai đầu, và cả `(sr, sc) → (sr, tc)` (cùng hàng) lẫn `(sr, tc) → (tr, tc)` (cùng cột) đều là **một nước**. Bàn cờ trống nên không có gì cản. Và không thể là 1 (do giả thiết), nên đúng 2.

Tức là chỉ cần xét thứ tự `0 → 1 → 2`: một dãy `if` ngắn, `O(1)` thời gian và bộ nhớ.

In [9]:
def classify(source, target):
    sr, sc = source
    tr, tc = target
    if sr == tr and sc == tc:
        return 0, "cùng ô"
    if sr == tr:
        return 1, "cùng hàng"
    if sc == tc:
        return 1, "cùng cột"
    if abs(sr - tr) == abs(sc - tc):
        return 1, "cùng đường chéo"
    return 2, "không cùng đường: đi qua góc chữ L"


for source, target in [([1, 1], [1, 1]), ([4, 2], [4, 7]), ([2, 5], [8, 5]), ([8, 1], [1, 8]), ([4, 2], [1, 3])]:
    print(source, "->", target, ":", classify(source, target))

[1, 1] -> [1, 1] : (0, 'cùng ô')
[4, 2] -> [4, 7] : (1, 'cùng hàng')
[2, 5] -> [8, 5] : (1, 'cùng cột')
[8, 1] -> [1, 8] : (1, 'cùng đường chéo')
[4, 2] -> [1, 3] : (2, 'không cùng đường: đi qua góc chữ L')


## 7. Các cách tiếp theo

### Cách 2 — Thử mọi ô trung gian

**Giải thích.** Không cần biết quy luật "góc chữ L": chỉ cần biết "tới được trong một nước" (`one_move`). Đáp án là:

1. `0` nếu hai ô trùng nhau;
2. `1` nếu `one_move(source, target)`;
3. `2` nếu tồn tại ô trung gian `m` mà `one_move(source, m)` và `one_move(m, target)` (thử mọi `m` trên 64 ô);
4. (phòng thủ) ngược lại thì thử tiếp độ sâu lớn hơn — nhưng sẽ chứng minh không xảy ra.

In [10]:
def min_moves_enumerate(source: list[int], target: list[int]) -> int:
    s, t = tuple(source), tuple(target)
    if s == t:
        return 0
    if one_move(s, t):
        return 1
    for r in range(1, 9):
        for c in range(1, 9):
            m = (r, c)
            if m != s and m != t and one_move(s, m) and one_move(m, t):
                return 2
    return 3                         # không bao giờ chạy tới đây (xem phần "Vì sao đúng")


for source, target, expected in examples:
    print(f"source={source}, target={target} -> {min_moves_enumerate(source, target)} (kỳ vọng {expected})")

source=[8, 1], target=[1, 8] -> 1 (kỳ vọng 1)
source=[4, 2], target=[1, 3] -> 2 (kỳ vọng 2)
source=[1, 1], target=[1, 1] -> 0 (kỳ vọng 0)


**Độ phức tạp Cách 2:** **Time `O(N²)`** với `N = 8` (thử `N²` ô trung gian, mỗi ô `O(1)`), tức hằng số `64` bước; **Space `O(1)`**.

### Cách 3 — Phân tích trường hợp (cách của lời giải tham khảo)

**Giải thích.** Viết thẳng ba trường hợp theo thứ tự: cùng ô → `0`; cùng hàng/cột/chéo → `1`; còn lại `2`. **Thứ tự kiểm tra rất quan trọng:** trường hợp `0` phải đứng **trước** vì hai ô trùng nhau cũng "cùng hàng" nên nếu kiểm tra "cùng hàng" trước sẽ trả `1` sai.

In [11]:
def min_moves_cases(source: list[int], target: list[int]) -> int:
    sr, sc = source
    tr, tc = target
    if sr == tr and sc == tc:
        return 0
    if sr == tr or sc == tc or abs(sr - tr) == abs(sc - tc):
        return 1
    return 2


for source, target, expected in examples:
    print(f"source={source}, target={target} -> {min_moves_cases(source, target)} (kỳ vọng {expected})")

source=[8, 1], target=[1, 8] -> 1 (kỳ vọng 1)
source=[4, 2], target=[1, 3] -> 2 (kỳ vọng 2)
source=[1, 1], target=[1, 1] -> 0 (kỳ vọng 0)


In [12]:
# Code nộp LeetCode cho Cách 3
from typing import List


class Solution:
    def minQueenMoves(self, source: List[int], target: List[int]) -> int:
        sr, sc = source
        tr, tc = target
        if sr == tr and sc == tc:
            return 0
        if sr == tr or sc == tc or abs(sr - tr) == abs(sc - tc):
            return 1
        return 2

**Độ phức tạp Cách 3:** **Time `O(1)`**, **Space `O(1)`**.

### Cách 4 — Biểu thức một dòng

**Giải thích.** Đặt `same = (source == target)` và `aligned = sr == tr or sc == tc or abs(sr − tr) == abs(sc − tc)`. Đáp án `= 0` nếu `same`; ngược lại `1 + (not aligned)` (cộng `True` là `1`). Hoặc: `int(not same) + int(not same and not aligned)`.

In [13]:
def min_moves_oneliner(source: list[int], target: list[int]) -> int:
    sr, sc = source
    tr, tc = target
    same = sr == tr and sc == tc
    aligned = sr == tr or sc == tc or abs(sr - tr) == abs(sc - tc)
    return 0 if same else 1 + (not aligned)


for source, target, expected in examples:
    print(f"source={source}, target={target} -> {min_moves_oneliner(source, target)} (kỳ vọng {expected})")

source=[8, 1], target=[1, 8] -> 1 (kỳ vọng 1)
source=[4, 2], target=[1, 3] -> 2 (kỳ vọng 2)
source=[1, 1], target=[1, 1] -> 0 (kỳ vọng 0)


**Độ phức tạp Cách 4:** **Time `O(1)`**, **Space `O(1)`**.

> Cách 1 là chuẩn đối chiếu "không cần suy luận". Cách 2 cho thấy ý tưởng ô trung gian. Cách 3 và Cách 4 là cách nộp bài thật sự (ngắn, `O(1)`).

## 8. TRACE từng bước

Ta theo dõi (a) BFS theo từng tầng cho `[4,2] → [1,3]`, và (b) Cách 3 qua ba ví dụ của đề.

In [14]:
def trace_bfs(source, target):
    start, goal = tuple(source), tuple(target)
    dist = {start: 0}
    level = [start]
    depth = 0
    print(f"BFS từ {start} tới {goal}")
    while level:
        print(f"  tầng {depth}: {len(level)} ô; có đích? {goal in level}")
        if goal in level:
            print(f"  => tìm thấy đích sau {depth} nước")
            return depth
        nxt = []
        for r, c in level:
            for nr, nc in one_move_squares(r, c):
                if (nr, nc) not in dist:
                    dist[(nr, nc)] = depth + 1
                    nxt.append((nr, nc))
        level = nxt
        depth += 1
    return -1


assert trace_bfs([4, 2], [1, 3]) == 2
assert trace_bfs([8, 1], [1, 8]) == 1

BFS từ (4, 2) tới (1, 3)
  tầng 0: 1 ô; có đích? False
  tầng 1: 23 ô; có đích? False
  tầng 2: 40 ô; có đích? True
  => tìm thấy đích sau 2 nước
BFS từ (8, 1) tới (1, 8)
  tầng 0: 1 ô; có đích? False
  tầng 1: 21 ô; có đích? True
  => tìm thấy đích sau 1 nước


In [15]:
def trace_cases(source, target):
    sr, sc = source
    tr, tc = target
    dr, dc = tr - sr, tc - sc
    print(f"source={source}, target={target}: dr={dr}, dc={dc}")
    if sr == tr and sc == tc:
        print("  cùng ô -> 0")
        return 0
    checks = [("cùng hàng (sr == tr)", sr == tr),
              ("cùng cột (sc == tc)", sc == tc),
              (f"cùng đường chéo (|dr|={abs(dr)} == |dc|={abs(dc)})", abs(dr) == abs(dc))]
    for name, ok in checks:
        print(f"  {name}: {ok}")
    if any(ok for _, ok in checks):
        print("  -> 1")
        return 1
    print(f"  không cùng đường -> 2 (qua góc ({sr}, {tc}))")
    return 2


for source, target, expected in examples:
    assert trace_cases(source, target) == expected

source=[8, 1], target=[1, 8]: dr=-7, dc=7
  cùng hàng (sr == tr): False
  cùng cột (sc == tc): False
  cùng đường chéo (|dr|=7 == |dc|=7): True
  -> 1
source=[4, 2], target=[1, 3]: dr=-3, dc=1
  cùng hàng (sr == tr): False
  cùng cột (sc == tc): False
  cùng đường chéo (|dr|=3 == |dc|=1): False
  không cùng đường -> 2 (qua góc (4, 3))
source=[1, 1], target=[1, 1]: dr=0, dc=0
  cùng ô -> 0


Bảng tóm tắt (Cách 3):

| source | target | `dr, dc` | hàng? | cột? | chéo? | đáp án |
|---|---|---|---|---|---|---|
| `(8, 1)` | `(1, 8)` | `-7, 7` | không | không | `|−7| = |7|` có | `1` |
| `(4, 2)` | `(1, 3)` | `-3, 1` | không | không | `3 ≠ 1` không | `2` |
| `(1, 1)` | `(1, 1)` | `0, 0` | (đã cùng ô) | | | `0` |

BFS khớp: tầng 0 chỉ có ô xuất phát; tầng 1 gồm mọi ô cùng hàng, cột hoặc đường chéo với nó; đích `(1, 3)` chỉ xuất hiện ở tầng 2.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (phân tích trường hợp). Đúng class `Solution`, đúng method `minQueenMoves(self, source: List[int], target: List[int]) -> int`.

In [16]:
from typing import List


class Solution:
    def minQueenMoves(self, source: List[int], target: List[int]) -> int:
        sr, sc = source
        tr, tc = target
        if sr == tr and sc == tc:                                 # đã ở đích
            return 0
        if sr == tr or sc == tc or abs(sr - tr) == abs(sc - tc):  # cùng hàng, cột hoặc đường chéo
            return 1
        return 2                                                  # đi qua ô (sr, tc)


for source, target, expected in examples:
    assert Solution().minQueenMoves(source, target) == expected
print("PASS ví dụ đề cho class Solution")

PASS ví dụ đề cho class Solution


## 10. Vì sao thuật toán đúng

- **Đáp án 0:** chỉ khi hai ô trùng nhau (không cần di chuyển).
- **Đáp án 1:** quân Hậu tới `target` trong một nước khi và chỉ khi `target` khác `source` và nằm trên cùng hàng, cùng cột hoặc cùng đường chéo với `source`. Đường chéo ⇔ `|dr| = |dc|` (đi `t` ô theo chéo thì hàng và cột đổi đúng `t`). Bàn cờ trống nên không bị chặn.
- **Đáp án luôn ≤ 2:** nếu không cùng đường, đi `source → (sr, tc) → target`: nước đầu cùng hàng `sr`, nước hai cùng cột `tc`. Cả hai nước đều hợp lệ (ô trung gian `(sr, tc)` nằm trong bàn cờ vì `1 ≤ sr, tc ≤ 8`) và đều dài ít nhất một ô (`sc ≠ tc` và `sr ≠ tr`).
- **Không thể đúng 1** trong nhánh cuối vì nó chính là phủ định của điều kiện 1; **không thể 0** vì hai ô khác nhau. Vậy là `2`.
- **Kiểm chứng mạnh:** vì bàn cờ chỉ có `64 × 64 = 4096` cặp (nguồn, đích), ta kiểm tra **tất cả** cặp đối chiếu với BFS ở mục test.

## 11. Phân tích Time / Space Complexity

Gọi `N = 8` là cạnh bàn cờ (đề cố định `8`).

- **Cách 1 (BFS):** Time `O(N³)`, Space `O(N²)`.
- **Cách 2 (thử ô trung gian):** Time `O(N²)`, Space `O(1)`.
- **Cách 3, Cách 4 (phân tích trường hợp):** Time **`O(1)`**, Space **`O(1)`**.
- Vì bàn cờ có kích thước cố định `8`, về mặt hình thức mọi cách đều là `O(1)`; ý nghĩa của phân tích trên là mức độ tổng quát cho bàn cờ `N × N`: nhánh cuối vẫn là `2` và Cách 3 vẫn `O(1)`.

## 12. Bộ test

Vì bàn cờ nhỏ, ta kiểm tra **mọi** cặp ô: BFS (Cách 1) là oracle độc lập, so với các cách còn lại và với lớp `Solution`.

In [17]:
sol = Solution()
impls = {
    "bfs": min_moves_bfs,
    "enumerate": min_moves_enumerate,
    "cases": min_moves_cases,
    "oneliner": min_moves_oneliner,
    "submit": sol.minQueenMoves,
}

for source, target, expected in examples:
    for name, fn in impls.items():
        assert fn(source, target) == expected, (name, source, target, fn(source, target), expected)
print("PASS ví dụ của đề (mọi cách)")

squares = [(r, c) for r in range(1, 9) for c in range(1, 9)]
histogram = {0: 0, 1: 0, 2: 0}
for s in squares:
    for t in squares:
        expected = min_moves_bfs(list(s), list(t))          # oracle độc lập (BFS)
        histogram[expected] += 1
        for name, fn in impls.items():
            got = fn(list(s), list(t))
            assert got == expected, (name, s, t, got, expected)
print("PASS toàn bộ 4096 cặp ô: mọi cách khớp BFS")
print("phân phối đáp án:", histogram)
assert histogram[0] == 64 and sum(histogram.values()) == 4096 and set(histogram) == {0, 1, 2}

PASS ví dụ của đề (mọi cách)


PASS toàn bộ 4096 cặp ô: mọi cách khớp BFS
phân phối đáp án: {0: 64, 1: 1456, 2: 2576}


### 12.1. Các trường hợp biên

In [18]:
def check_all(source, target, expected, label):
    for name, fn in impls.items():
        got = fn(source, target)
        assert got == expected, (label, name, got, expected)
    print("PASS", label, "->", expected)


check_all([1, 1], [8, 8], 1, "hai góc đối diện cùng đường chéo chính")
check_all([1, 8], [8, 1], 1, "hai góc đối diện cùng đường chéo phụ")
check_all([1, 1], [1, 8], 1, "cùng hàng, hai góc")
check_all([1, 1], [8, 1], 1, "cùng cột, hai góc")
check_all([1, 1], [2, 3], 2, "kiểu quân Mã: không cùng đường")
check_all([8, 8], [8, 8], 0, "cùng ô ở góc")
check_all([4, 4], [4, 4], 0, "cùng ô ở giữa")
check_all([1, 2], [2, 1], 1, "hai ô kề chéo ở góc (chéo phụ)")
check_all([1, 2], [3, 3], 2, "(1,2) -> (3,3): |dr| = 2, |dc| = 1")
check_all([3, 3], [3, 4], 1, "hai ô kề ngang")
check_all([2, 7], [7, 3], 2, "xa nhau, không cùng đường")

PASS hai góc đối diện cùng đường chéo chính -> 1
PASS hai góc đối diện cùng đường chéo phụ -> 1
PASS cùng hàng, hai góc -> 1
PASS cùng cột, hai góc -> 1
PASS kiểu quân Mã: không cùng đường -> 2
PASS cùng ô ở góc -> 0
PASS cùng ô ở giữa -> 0
PASS hai ô kề chéo ở góc (chéo phụ) -> 1
PASS (1,2) -> (3,3): |dr| = 2, |dc| = 1 -> 2
PASS hai ô kề ngang -> 1
PASS xa nhau, không cùng đường -> 2


### 12.2. Tính chất và phản ví dụ

In [19]:
# Tính chất 1: đối xứng, dist(a, b) == dist(b, a)
random.seed(4060)
for _ in range(500):
    a = [random.randint(1, 8), random.randint(1, 8)]
    b = [random.randint(1, 8), random.randint(1, 8)]
    assert sol.minQueenMoves(a, b) == sol.minQueenMoves(b, a)
print("PASS tính chất: đối xứng")

# Tính chất 2: bất đẳng thức tam giác  d(a, c) <= d(a, b) + d(b, c)
random.seed(4061)
for _ in range(2000):
    a, b, c = ([random.randint(1, 8), random.randint(1, 8)] for _ in range(3))
    assert sol.minQueenMoves(a, c) <= sol.minQueenMoves(a, b) + sol.minQueenMoves(b, c)
print("PASS tính chất: bất đẳng thức tam giác")

# Tính chất 3: số ô cách đúng 1 nước từ góc (1,1) là 21; từ (4,4) là 27
def count_dist_one(s):
    return sum(1 for t in squares if sol.minQueenMoves(list(s), list(t)) == 1)

assert count_dist_one((1, 1)) == 21 and count_dist_one((4, 4)) == 27
print("PASS tính chất: số ô cách 1 nước (góc = 21, giữa = 27)")

# Phản ví dụ 1: coi là bài của VUA (đi 1 ô mỗi nước): khoảng cách Chebyshev
def sai_vua(source, target):
    return max(abs(source[0] - target[0]), abs(source[1] - target[1]))

print("(8,1)->(1,8): sai (vua):", sai_vua([8, 1], [1, 8]), "| đúng:", sol.minQueenMoves([8, 1], [1, 8]))
assert sai_vua([8, 1], [1, 8]) == 7 and sol.minQueenMoves([8, 1], [1, 8]) == 1

PASS tính chất: đối xứng
PASS tính chất: bất đẳng thức tam giác
PASS tính chất: số ô cách 1 nước (góc = 21, giữa = 27)
(8,1)->(1,8): sai (vua): 7 | đúng: 1


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Bàn cờ đã được duyệt hết, nhưng ta vẫn thử một lần ngẫu nhiên có seed (tiện lợi để thấy thói quen kiểm tra nhiều input) và thêm bàn cờ lớn hơn `N × N` cho BFS tổng quát để thấy quy luật `0/1/2` vẫn đúng.

In [20]:
random.seed(2026)
for trial in range(3000):
    s = [random.randint(1, 8), random.randint(1, 8)]
    t = [random.randint(1, 8), random.randint(1, 8)]
    expected = min_moves_bfs(s, t)
    for name, fn in impls.items():
        assert fn(s, t) == expected, (name, s, t)


def bfs_general(size, s, t):
    """BFS cho bàn cờ size x size."""
    s, t = tuple(s), tuple(t)
    dist = {s: 0}
    q = deque([s])
    while q:
        r, c = q.popleft()
        if (r, c) == t:
            return dist[(r, c)]
        for dr, dc in DIRECTIONS:
            nr, nc = r + dr, c + dc
            while 1 <= nr <= size and 1 <= nc <= size:
                if (nr, nc) not in dist:
                    dist[(nr, nc)] = dist[(r, c)] + 1
                    q.append((nr, nc))
                nr, nc = nr + dr, nc + dc
    return -1


for size in [2, 3, 5, 10, 13]:
    for _ in range(200):
        s = [random.randint(1, size), random.randint(1, size)]
        t = [random.randint(1, size), random.randint(1, size)]
        assert bfs_general(size, s, t) == min_moves_cases(s, t)
print("PASS 3000 input ngẫu nhiên trên bàn 8x8 và bàn cờ N x N (N = 2..13): cùng quy luật 0/1/2")

PASS 3000 input ngẫu nhiên trên bàn 8x8 và bàn cờ N x N (N = 2..13): cùng quy luật 0/1/2


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên trường hợp `0` (cùng ô)

In [21]:
def sai_quen_khong(source, target):
    sr, sc = source
    tr, tc = target
    if sr == tr or sc == tc or abs(sr - tr) == abs(sc - tc):
        return 1
    return 2


print("sai:", sai_quen_khong([1, 1], [1, 1]), "| đúng:", Solution().minQueenMoves([1, 1], [1, 1]))

sai: 1 | đúng: 0


**Vì sao sai:** hai ô trùng nhau cũng "cùng hàng", "cùng cột", "cùng đường chéo" (`|0| == |0|`), nên nếu kiểm tra các điều kiện đó trước thì trả `1`. Phải kiểm tra `source == target` **trước tiên** và trả `0`.

### Lỗi 2: Chỉ xét một đường chéo

In [22]:
def sai_mot_cheo(source, target):
    sr, sc = source
    tr, tc = target
    if sr == tr and sc == tc:
        return 0
    if sr == tr or sc == tc or sr - sc == tr - tc:        # chỉ có đường chéo chính
        return 1
    return 2


print("sai:", sai_mot_cheo([8, 1], [1, 8]), "| đúng:", Solution().minQueenMoves([8, 1], [1, 8]))

sai: 2 | đúng: 1


**Vì sao sai:** có **hai** họ đường chéo: `sr − sc` không đổi (chéo chính) và `sr + sc` không đổi (chéo phụ). Ví dụ 1 `(8,1) → (1,8)` nằm trên chéo phụ (`8 + 1 = 1 + 8`) nên `sr − sc ≠ tr − tc` (`7` và `-7`). Dùng `abs(sr − tr) == abs(sc − tc)` gộp cả hai.

### Lỗi 3: Nhầm sang cách đi của quân Vua hoặc khoảng cách Manhattan

In [23]:
def sai_manhattan(source, target):
    return abs(source[0] - target[0]) + abs(source[1] - target[1])


def sai_chebyshev(source, target):
    return max(abs(source[0] - target[0]), abs(source[1] - target[1]))


for s, t, exp in examples:
    print(s, t, "| Manhattan:", sai_manhattan(s, t), "| Chebyshev (vua):", sai_chebyshev(s, t), "| đúng:", exp)

[8, 1] [1, 8] | Manhattan: 14 | Chebyshev (vua): 7 | đúng: 1
[4, 2] [1, 3] | Manhattan: 4 | Chebyshev (vua): 3 | đúng: 2
[1, 1] [1, 1] | Manhattan: 0 | Chebyshev (vua): 0 | đúng: 0


**Vì sao sai:** quân Hậu đi **nhiều ô** trong một nước, nên "số ô" không còn ý nghĩa; chỉ quan trọng là **có cùng đường hay không**. Manhattan đếm số bước ngang/dọc 1 ô, Chebyshev đếm số bước của quân Vua. Với ví dụ 1, Manhattan ra `14` và Chebyshev ra `7`, trong khi đáp án đúng là `1`.

### Lỗi 4: Nghĩ rằng đáp án có thể lớn hơn 2 và tự viết BFS rườm rà hoặc trả `-1`

In [24]:
def sai_tra_tru_1(source, target):
    sr, sc = source
    tr, tc = target
    if sr == tr and sc == tc:
        return 0
    if sr == tr or sc == tc or abs(sr - tr) == abs(sc - tc):
        return 1
    # tưởng rằng nếu không cùng đường thì quân Hậu "không đi tới được" trong 2 nước
    return -1


print("sai:", sai_tra_tru_1([4, 2], [1, 3]), "| đúng:", Solution().minQueenMoves([4, 2], [1, 3]))

sai: -1 | đúng: 2


**Vì sao sai:** bàn cờ trống nên luôn có "góc chữ L" `(sr, tc)` (hoặc `(tr, sc)`) làm ô trung gian. Mọi cặp ô đều tới được nhau trong **tối đa 2 nước**; không bao giờ phải trả `-1` hay `3`. Hãy tự kiểm tra bằng BFS trên mọi cặp ô như phần test ở trên.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Đề hỏi "**số nước đi ít nhất**" trên bàn cờ nhỏ, không có chướng ngại: nghĩ ngay tới **BFS** để có đáp án chuẩn, rồi **quan sát quy luật** (ở đây chỉ `0/1/2`).
- Quân **Hậu** = Xe + Tượng. Nhớ: Xe (cùng hàng/cột) tới mọi ô trong 2 nước (qua góc chữ L); Tượng (chéo) bị giới hạn bởi màu ô.
- Điều kiện chéo: `abs(dr) == abs(dc)`. Luôn nhớ **hai** hướng chéo.
- Kiểm tra `source == target` **trước** các điều kiện "cùng đường".
- Bàn cờ `8 × 8` chỉ có `4096` cặp: **kiểm tra toàn bộ** thay vì vài ví dụ; đây là cách tự tin nhất khi không chắc quy luật.
- Khi có chướng ngại (quân cản) quy luật `0/1/2` không còn đúng và cần BFS thật sự.

## 15. Biến thể của bài

1. **Quân Xe (rook):** 0 nếu cùng ô, 1 nếu cùng hàng/cột, còn lại 2.
2. **Quân Tượng (bishop):** `-1` nếu khác màu ô (`(r + c) % 2` khác nhau); `0` nếu cùng ô; `1` nếu cùng đường chéo; ngược lại `2`.
3. **Quân Mã (knight):** BFS thật sự (LeetCode 1197, Minimum Knight Moves).
4. **Bàn cờ `N × N`:** vẫn `0/1/2` cho quân Hậu khi `N ≥ 1`.
5. **Có chướng ngại** (một số ô bị chặn): cần BFS.

In [25]:
def rook_moves(source, target):
    """Biến thể 1."""
    if source == target:
        return 0
    return 1 if source[0] == target[0] or source[1] == target[1] else 2


def bishop_moves(source, target):
    """Biến thể 2: -1 nếu khác màu."""
    sr, sc = source
    tr, tc = target
    if (sr + sc) % 2 != (tr + tc) % 2:
        return -1
    if source == target:
        return 0
    return 1 if abs(sr - tr) == abs(sc - tc) else 2


def knight_moves(source, target):
    """Biến thể 3: BFS với 8 nước nhảy chữ L."""
    jumps = [(1, 2), (2, 1), (-1, 2), (-2, 1), (1, -2), (2, -1), (-1, -2), (-2, -1)]
    s, t = tuple(source), tuple(target)
    dist = {s: 0}
    q = deque([s])
    while q:
        r, c = q.popleft()
        if (r, c) == t:
            return dist[(r, c)]
        for dr, dc in jumps:
            nr, nc = r + dr, c + dc
            if 1 <= nr <= 8 and 1 <= nc <= 8 and (nr, nc) not in dist:
                dist[(nr, nc)] = dist[(r, c)] + 1
                q.append((nr, nc))
    return -1


def bfs_with_directions(source, target, directions):
    """BFS trượt theo các hướng cho trước (kiểm chứng xe/tượng)."""
    s, t = tuple(source), tuple(target)
    dist = {s: 0}
    q = deque([s])
    while q:
        r, c = q.popleft()
        if (r, c) == t:
            return dist[(r, c)]
        for dr, dc in directions:
            nr, nc = r + dr, c + dc
            while 1 <= nr <= 8 and 1 <= nc <= 8:
                if (nr, nc) not in dist:
                    dist[(nr, nc)] = dist[(r, c)] + 1
                    q.append((nr, nc))
                nr, nc = nr + dr, nc + dc
    return -1


ROOK_DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
BISHOP_DIRS = [(-1, -1), (-1, 1), (1, -1), (1, 1)]
for s in squares:
    for t in squares:
        assert rook_moves(list(s), list(t)) == bfs_with_directions(s, t, ROOK_DIRS)
        assert bishop_moves(list(s), list(t)) == bfs_with_directions(s, t, BISHOP_DIRS)
assert knight_moves([1, 1], [2, 3]) == 1 and knight_moves([1, 1], [8, 8]) == 6
print("PASS biến thể: xe, tượng khớp BFS trên mọi cặp ô; mã dùng BFS")

PASS biến thể: xe, tượng khớp BFS trên mọi cặp ô; mã dùng BFS


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS trên bàn cờ | đường đi ngắn nhất theo luật đi | `O(N³)` | `O(N²)` | **oracle** đối chiếu; khi có chướng ngại |
| 2. Thử ô trung gian | `0`, `1`, hoặc tồn tại `m` với 2 nước | `O(N²)` | `O(1)` | thấy ý "đi qua ô trung gian" |
| 3. Phân tích trường hợp | `0` → `1` (cùng đường) → `2` | `O(1)` | `O(1)` | **cách nộp bài** |
| 4. Biểu thức một dòng | `0 if same else 1 + (not aligned)` | `O(1)` | `O(1)` | ngắn gọn khi đã hiểu |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `abs(3 - 7)`, `abs(2 - 5) == abs(7 - 4)`, `Solution().minQueenMoves([1, 1], [8, 8])`, `Solution().minQueenMoves([1, 1], [2, 3])`, `Solution().minQueenMoves([3, 3], [3, 3])`.

**Bài 2 (tính tay).** Với `source = [5, 5]`, `target = [1, 2]`: tính `dr`, `dc`, cho biết có cùng đường không, đáp án là gì và chỉ ra một ô trung gian khi cần. Làm lại với `source = [2, 6]`, `target = [5, 3]`.

**Bài 3 (điền TODO).** Hoàn thành `queen_todo(source, target)` theo Cách 3.

In [26]:
def queen_todo(source, target):
    sr, sc = source
    tr, tc = target
    # TODO 1: nếu hai ô trùng nhau trả về 0
    # TODO 2: nếu cùng hàng, cùng cột hoặc abs(sr - tr) == abs(sc - tc) trả về 1
    # TODO 3: còn lại trả về 2
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `queen_bfs2(source, target)` theo Cách 1 (BFS bằng `deque`) mà không nhìn code mẫu. Kiểm tra bằng ba ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `count_reachable_in_one(source)` đếm số ô tới được **trong đúng một nước** (không tính ô xuất phát) trên bàn `8 × 8` trống. (b) Viết `squares_at_distance_two(source)` trả về số ô có khoảng cách **đúng 2** từ `source`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `abs` trả về giá trị không âm; hai góc đối diện của bàn cờ cùng một đường chéo; `(1,1) → (2,3)` có `dr = 1`, `dc = 2`. Bài 2: tính `dr`, `dc`, rồi so sánh `|dr|` và `|dc|`.
- **Mức 2 (vừa):** Bài 1: `4`, `True`, `1`, `2`, `0`. Bài 2: `[5,5] → [1,2]`: `dr = -4`, `dc = -3`, không cùng hàng, cột hay chéo (`4 ≠ 3`) nên đáp án `2` qua `(5, 2)` hoặc `(1, 5)`; `[2,6] → [5,3]`: `dr = 3`, `dc = -3` nên cùng chéo, đáp án `1`. Bài 3: ba TODO trong comment.
- **Mức 3 (sâu):** Bài 5(a): đếm các `t != source` với `one_move(source, t)`; từ `(1,1)` là `21`, từ `(4,4)` là `27`. Bài 5(b): `64 − 1 − count_reachable_in_one(source)` vì mọi ô còn lại cách đúng 2 (không có ô nào cách 3).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [27]:
# Bài 1
assert abs(3 - 7) == 4
assert (abs(2 - 5) == abs(7 - 4)) is True
assert Solution().minQueenMoves([1, 1], [8, 8]) == 1
assert Solution().minQueenMoves([1, 1], [2, 3]) == 2
assert Solution().minQueenMoves([3, 3], [3, 3]) == 0

# Bài 2
assert (1 - 5, 2 - 5) == (-4, -3)
assert Solution().minQueenMoves([5, 5], [1, 2]) == 2
assert one_move([5, 5], [5, 2]) and one_move([5, 2], [1, 2])
assert Solution().minQueenMoves([2, 6], [5, 3]) == 1

# Bài 3
def queen_todo_ans(source, target):
    sr, sc = source
    tr, tc = target
    if sr == tr and sc == tc:
        return 0
    if sr == tr or sc == tc or abs(sr - tr) == abs(sc - tc):
        return 1
    return 2

for source, target, expected in examples:
    assert queen_todo_ans(source, target) == expected

# Bài 4
def queen_bfs2(source, target):
    s, t = tuple(source), tuple(target)
    dist = {s: 0}
    q = deque([s])
    while q:
        r, c = q.popleft()
        if (r, c) == t:
            return dist[(r, c)]
        for dr, dc in DIRECTIONS:
            nr, nc = r + dr, c + dc
            while 1 <= nr <= 8 and 1 <= nc <= 8:
                if (nr, nc) not in dist:
                    dist[(nr, nc)] = dist[(r, c)] + 1
                    q.append((nr, nc))
                nr, nc = nr + dr, nc + dc
    return -1

for source, target, expected in examples:
    assert queen_bfs2(source, target) == expected

# Bài 5(a), 5(b)
def count_reachable_in_one(source):
    s = tuple(source)
    return sum(1 for t in squares if t != s and one_move(s, t))

def squares_at_distance_two(source):
    s = tuple(source)
    return sum(1 for t in squares if Solution().minQueenMoves(list(s), list(t)) == 2)

assert count_reachable_in_one([1, 1]) == 21 and count_reachable_in_one([4, 4]) == 27
assert squares_at_distance_two([1, 1]) == 64 - 1 - 21
assert squares_at_distance_two([4, 4]) == 64 - 1 - 27
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (quân Hậu trên bàn cờ trống: `0 / 1 / 2`)

```python
class Solution:
    def minQueenMoves(self, source: List[int], target: List[int]) -> int:
        sr, sc = source
        tr, tc = target
        if sr == tr and sc == tc:
            return 0
        if sr == tr or sc == tc or abs(sr - tr) == abs(sc - tc):
            return 1
        return 2
```

**Checklist**

- [ ] Kiểm tra `source == target` **trước** (trả `0`)?
- [ ] "Cùng đường" gồm **hàng, cột và cả hai đường chéo** (`abs(dr) == abs(dc)`)?
- [ ] Nhánh cuối trả `2` (qua ô góc `(sr, tc)`), không bao giờ `-1` hay `3`?
- [ ] Không nhầm với Vua (Chebyshev) hoặc khoảng cách Manhattan?
- [ ] Đã thử: hai góc đối diện, hai ô kề chéo, kiểu "chữ L" của quân Mã, cùng ô?
- [ ] Đã đối chiếu với BFS trên toàn bộ 4096 cặp ô?
- [ ] Skeleton đúng `class Solution` / `def minQueenMoves(self, source: List[int], target: List[int]) -> int`, có `from typing import List`?